# One watermark, three file formats

A verifier may receive the suspect model as `.pt`, `.onnx` or a Keras file. The SDK reads the same layer from each and gets the identical quantised vector, hence the same `w_hash` that a proof is bound to.

In [ ]:
import gzip, numpy as np
from pathlib import Path
DATA = Path("../../demo/data")          # MNIST shipped with the repository
def idx(name):
    raw = gzip.open(DATA / name, "rb").read(); nd = raw[3]
    dims = [int.from_bytes(raw[4 + 4*i: 8 + 4*i], "big") for i in range(nd)]
    return np.frombuffer(raw, dtype=np.uint8, offset=4 + 4*nd).reshape(dims)
x = ((idx("train-images-idx3-ubyte.gz")[:12000] / 255.0 - 0.1307) / 0.3081).astype("float32")
y = idx("train-labels-idx1-ubyte.gz")[:12000].astype("int64")
xt = ((idx("t10k-images-idx3-ubyte.gz")[:2000] / 255.0 - 0.1307) / 0.3081).astype("float32")
yt = idx("t10k-labels-idx1-ubyte.gz")[:2000].astype("int64")
print(x.shape, xt.shape)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from sigil import protocol as P
from sigil.embed import embed_torch
from sigil.adapters import quantized_vector

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1, self.conv2 = nn.Conv2d(1, 32, 3), nn.Conv2d(32, 64, 3)
        self.fc1, self.fc2 = nn.Linear(9216, 128), nn.Linear(128, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv2(F.relu(self.conv1(x)))), 2)
        return self.fc2(F.relu(self.fc1(torch.flatten(x, 1))))

cfg = P.Config()
seed, pkh = P.new_seed(), P.pk_hash(bytes(32))
torch.manual_seed(0)
model, det = embed_torch(Net(), (torch.from_numpy(x).unsqueeze(1), torch.from_numpy(y)), cfg, seed, pkh, epochs=1)
model = model.cpu().eval()
torch.save(model.state_dict(), "suspect.pt")
torch.onnx.export(model, (torch.zeros(1, 1, 28, 28),), "suspect.onnx", dynamo=False, do_constant_folding=False)
q_pt, q_onnx = quantized_vector("suspect.pt", cfg), quantized_vector("suspect.onnx", cfg)
print("identical vectors:", (q_pt == q_onnx).all())
print("w_hash .pt  :", P.w_hash(q_pt, cfg))
print("w_hash .onnx:", P.w_hash(q_onnx, cfg))
assert (q_pt == q_onnx).all()